[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/UrbanAccessAnalyzer/blob/main/examples/walkability.ipynb)

# 🚶 Walkability

**How many everyday places can you reach on foot?** Every street is graded by its walking distance to shops, schools, health care, parks...

- An **isochrone** is the set of streets reachable within a walking distance from a place. One is built around every place, in distance steps.
- A street scores **100** within the first step of a place, less and less up to the type's `max_distance`, and **0** beyond. The grades of all types are averaged, weighted by `importance`.
- **Parks are polygons**: people enter anywhere along the edge, so streets touching or inside a park count as being at it.

*Everything in one call:* `workflows.walkability(place, pois)`.

In [ ]:
# Google Colab only: installs everything (all packages pinned to release v1.0.0). Does nothing locally.
import sys

if "google.colab" in sys.modules:
    !pip install -q "UrbanAccessAnalyzer[plot] @ git+https://github.com/CityScope/UrbanAccessAnalyzer.git@v1.0.0" "geohierarchy @ git+https://github.com/CityScope/geohierarchy.git@v1.0.0" "pycensus @ git+https://github.com/CityScope/pyCensus.git@v1.0.0"
    # If an import fails afterwards: Runtime > Restart session, then run again.

In [ ]:
import sys

from UrbanAccessAnalyzer import api, grading, plotting, select_aoi, utils, workflows

# 1 Area of interest

Write a **place name** (its boundary is drawn on the map) or **draw your own polygon or rectangle** with the tools on the left. If you draw nothing, the place is used.

In [ ]:
# USER INPUT
place = "Alcoy, España"  # a place name; or draw your own area on the map below (then this only names the output folder)
folder = f"output/{utils.sanitize_filename(place)}"  # results are saved here
cache_dir = "cache"  # downloads (place, streets, population) are kept here, next to the notebook, and reused when you rerun

selector = select_aoi(place, cache_dir=cache_dir)  # searches the place
selector.map  # the place, dashed blue. Draw a polygon or rectangle here to use your own area instead

In [ ]:
aoi = selector.aoi  # the area used from here on: what you drew, or else the place

# Settings

In [ ]:
# USER INPUT
download_buffer = 1500  # meters of extra area for streets and places (at least the largest max_distance)
n_steps = 5  # distance steps per type of place: with 5, a street scores 100, 80, 60, 40 or 20, then 0
h3_resolution = 10  # hexagon size of the results: 10 is ~0.015 km2, 9 is ~0.1 km2
population = "worldpop"  # "worldpop": 100 m grid, anywhere in the world | "census": the country's own census (an error if not implemented)
empty_cells = "nearest_value"  # hexagons with people but no street: "nearest_value" (copy the nearest hexagon's value) | "relocate_weight" (move their people to the nearest one, within 300 m) | "ignore" (no score)

## The places

One line per type of place: `grading.poi(name, source, importance=..., max_distance=..., icon=...)`. `importance` is the weight in the final score, `max_distance` the meters walked beyond which the place is worthless, `icon` the emoji on the maps. Ready-made `source`s: `"schools"`, `"groceries"`, `"restaurants"`, `"healthcare"`, `"shops"`, `"libraries"`, `"pharmacies"`, `"gyms"`, `"cinemas"`, `"bus_stops"` and `"green_areas"` (parks, polygons).

In [ ]:
# USER INPUT
pois = [
    grading.poi("schools",     "schools",     importance=10, max_distance=1000, icon="🏫"),
    grading.poi("groceries",   "groceries",   importance=8,  max_distance=1500, icon="🛒"),
    grading.poi("restaurants", "restaurants", importance=7,  max_distance=1500, icon="🍽️"),
    grading.poi("healthcare",  "healthcare",  importance=3,  max_distance=2000, icon="🏥"),
    grading.poi("parks",       "green_areas", importance=5,  max_distance=1500, icon="🌳"),
    grading.poi("bakeries",    ("shop", "bakery"), importance=4, max_distance=1000, icon="🥖"),  # any OSM tag
]

### Add your own places (optional)

`source` can also be an **OpenStreetMap tag**, a **file of your own** or an **Overpass filter**:

| `source` | Example |
|---|---|
| OSM tag `(key, value)` | `("amenity", "dentist")`; `"*"` = any value; find tags in the [map features](https://wiki.openstreetmap.org/wiki/Map_features) |
| your own file, points or polygons (`.gpkg`, `.geojson`, `.shp`, `.parquet`...) | `"my_places.gpkg"`; `name_column="..."` for the names |
| Overpass filter | `'nwr["amenity"~"hospital\|clinic"]'` |

In **Google Colab** your computer's files are not visible: if the file is not there, *a window opens to choose it from your computer*.

In [ ]:
# USER INPUT. Uncomment what you want: each line adds one type of place.
# pois.append(grading.poi("dentists", ("amenity", "dentist"), importance=3, max_distance=1000, icon="🦷"))      # an OSM tag
# pois.append(grading.poi("my places", "my_places.gpkg", name_column=None, importance=5, max_distance=1000, icon="⭐"))  # your file

For an **Overpass filter**, `nwr` means nodes, ways and relations and `~"a|b"` means "a or b". To get one from an AI assistant, ask:

> Modify this OpenStreetMap Overpass filter to search for `"hospital"` instead of pharmacies. Return only the filter, on one line, without bounding box or `out` statements: `nwr["amenity"="pharmacy"]`

In [ ]:
# USER INPUT (Overpass filter)
# pois.append(grading.poi("clinics", 'nwr["amenity"~"hospital|clinic"]', importance=3, max_distance=2000, icon="💊"))

# 2 Data download

The study area, the walking streets (OpenStreetMap, through Geofabrik extracts) and the places (OpenStreetMap through the Overpass API, or your files). Streets and places are downloaded for a *bigger* area (`download_buffer`), so those just outside still count. Downloads are kept in the `cache` folder and reused when you rerun. If Overpass is busy, run the cell again.

In [ ]:
aoi_download = utils.buffer_gdf(aoi, download_buffer)  # a bit bigger, for the download

network = api.StreetNetwork.from_pbf(f"{cache_dir}/pbf/region.osm.pbf", aoi=aoi_download, network_type="walk",
                                     simplify_distance=30, cache_dir=f"{cache_dir}/networks")
poi_gdfs = grading.fetch_pois(pois, aoi_download, crs=network.crs)  # one table of places per type
print(f"{network.nodes.height} street intersections, {network.edges.height} street segments")

In [ ]:
poi_table = grading.poi_table(pois, poi_gdfs)  # all the places: type, name, icon
plotting.save_poi_map(f"{folder}/maps/points_of_interest.html", aoi, poi_table)  # also saved as a standalone .html

# 3 Scoring the places

For every type of place: its distance steps (`max_distance` divided in `n_steps`), the score of a street inside each one (100 in the first, 0 beyond the last) and its share in the final score (`weight %`).

In [ ]:
grading.tiers_table(pois, n_steps)

# 4 Isochrones

From every place we walk along the streets up to `max_distance`: each street gets the score of the closest step, and the final score is the mean of all types weighted by `importance`.

In [ ]:
edges, network = grading.grade(network, pois, poi_gdfs, n_steps=n_steps, aoi=aoi)
edges.to_file(f"{folder}/walkability_streets.gpkg")
grading.score_summary(edges, pois)  # statistics of the scores (0-100) of every type of place

In [ ]:
show = "schools"  # any type of place above, to see how far it reaches along the streets
plotting.plot_streets(edges, show)

# 5 Results

In [ ]:
hexes, stats, m = workflows.walkability_results(edges, aoi, folder, pois, poi_table, h3_resolution, population, "walkability_map.html", empty_cells=empty_cells, cache_dir=cache_dir)
m

**Who benefits?** People living in each walkability band:

In [ ]:
stats  # also saved as stats.csv

In [ ]:
if "google.colab" in sys.modules:
    !zip -r /content/output.zip output  # then download it from the file browser

Saved in `output/<place>/`:

- `maps/points_of_interest.html` and `maps/walkability_map.html`: interactive maps, double-click to open them in your browser
- `walkability_streets.gpkg`: streets with one score per type of place and the final `access_score`
- `stats.csv`: people per walkability band